In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data        = load_all_data()
peer_groups = data['peer_groups']
sectors     = data['sectors']

# Load financial ratios
conn   = sqlite3.connect('data/nifty100.db')
ratios = pd.read_sql_query(
    "SELECT * FROM financial_ratios_computed", conn
)
conn.close()

print(f"Peer groups: {peer_groups.shape}")
print(f"Peer group names: {peer_groups['peer_group_name'].unique()}")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Peer groups: (56, 4)
Peer group names: ['Private Banks' 'Public Sector Banks' 'IT Services' 'Pharmaceuticals'
 'Automobiles' 'Life Insurance' 'Oil & Gas' 'Power & Utilities' 'Steel'
 'FMCG' 'Consumer Finance']


In [2]:
ANALYSIS_YEAR = '2024-03'

# Get latest ratios
latest = ratios[ratios['year'] == ANALYSIS_YEAR].copy()

# Force numeric types
metric_cols = [
    'return_on_equity_pct', 'return_on_capital_pct',
    'net_profit_margin_pct', 'debt_to_equity',
    'free_cash_flow_cr', 'net_profit_cagr_5yr',
    'sales_cagr_5yr', 'eps_cagr_5yr',
    'interest_coverage', 'asset_turnover'
]

for col in metric_cols:
    if col in latest.columns:
        latest[col] = pd.to_numeric(latest[col], errors='coerce')

# Merge peer group info
peer_df = pd.merge(
    peer_groups,
    latest,
    on='company_id',
    how='left'
)

print(f"Peer DataFrame shape: {peer_df.shape}")
print(f"Peer groups: {peer_df['peer_group_name'].nunique()}")
print(f"Companies in peer groups: {peer_df['company_id'].nunique()}")

Peer DataFrame shape: (56, 46)
Peer groups: 11
Companies in peer groups: 56


In [3]:
def compute_peer_percentiles(peer_df, metric_cols):
    """
    Computes percentile rank for each metric within each peer group.
    For D/E: inverts the rank (lower D/E = higher percentile).

    Returns:
        DataFrame: Long format with company_id, peer_group_name,
                   metric, value, percentile_rank
    """
    results = []

    for group_name, group in peer_df.groupby('peer_group_name'):
        for metric in metric_cols:
            if metric not in group.columns:
                continue

            # Compute percentile rank within group
            pct_rank = group[metric].rank(pct=True, na_option='keep')

            # Invert D/E rank (lower is better)
            if metric == 'debt_to_equity':
                pct_rank = 1 - pct_rank

            for _, row in group.iterrows():
                results.append({
                    'company_id':      row['company_id'],
                    'peer_group_name': group_name,
                    'metric':          metric,
                    'value':           round(row[metric], 2)
                                       if pd.notna(row[metric]) else None,
                    'percentile_rank': round(pct_rank[row.name], 3)
                                       if pd.notna(pct_rank[row.name]) else None,
                    'year':            ANALYSIS_YEAR,
                })

    return pd.DataFrame(results)

peer_percentiles = compute_peer_percentiles(peer_df, metric_cols)

print(f"Peer percentiles shape: {peer_percentiles.shape}")
print(f"\nSample — TCS percentiles:")
print(peer_percentiles[peer_percentiles['company_id'] == 'TCS'][
    ['company_id', 'peer_group_name', 'metric',
     'value', 'percentile_rank']
].to_string(index=False))

Peer percentiles shape: (560, 6)

Sample — TCS percentiles:
company_id peer_group_name                metric    value  percentile_rank
       TCS     IT Services  return_on_equity_pct    50.94              1.0
       TCS     IT Services return_on_capital_pct    60.21              1.0
       TCS     IT Services net_profit_margin_pct    19.14              1.0
       TCS     IT Services        debt_to_equity     0.09              0.5
       TCS     IT Services     free_cash_flow_cr 50429.00              1.0
       TCS     IT Services   net_profit_cagr_5yr     7.87              0.4
       TCS     IT Services        sales_cagr_5yr    10.46              0.4
       TCS     IT Services          eps_cagr_5yr     8.62              0.4
       TCS     IT Services     interest_coverage    87.10              0.8
       TCS     IT Services        asset_turnover     1.66              1.0


In [4]:
conn = sqlite3.connect('data/nifty100.db')
peer_percentiles.to_sql(
    'peer_percentiles', conn, if_exists='replace', index=False
)

cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM peer_percentiles")
count = cursor.fetchone()[0]
conn.close()

print(f"peer_percentiles table saved — {count} rows")
print(f"Peer groups covered: {peer_percentiles['peer_group_name'].nunique()}")
print(f"Companies covered: {peer_percentiles['company_id'].nunique()}")

peer_percentiles table saved — 560 rows
Peer groups covered: 11
Companies covered: 56


In [5]:
# Verify: in IT Services group, highest ROE company
# should have the highest ROE percentile rank
it_roe = peer_percentiles[
    (peer_percentiles['peer_group_name'] == 'IT Services') &
    (peer_percentiles['metric'] == 'return_on_equity_pct')
].sort_values('percentile_rank', ascending=False)

print("IT Services — ROE Rankings:")
print(it_roe[['company_id', 'value', 'percentile_rank']].to_string(index=False))

print()

# Also check FMCG group
fmcg_roe = peer_percentiles[
    (peer_percentiles['peer_group_name'] == 'FMCG') &
    (peer_percentiles['metric'] == 'return_on_equity_pct')
].sort_values('percentile_rank', ascending=False)

print("FMCG — ROE Rankings:")
print(fmcg_roe[['company_id', 'value', 'percentile_rank']].to_string(index=False))

IT Services — ROE Rankings:
company_id  value  percentile_rank
       TCS  50.94              1.0
      INFY  29.79              0.8
   HCLTECH  23.01              0.6
      LTIM  22.90              0.4
     TECHM   8.99              0.2

FMCG — ROE Rankings:
company_id  value  percentile_rank
 NESTLEIND 117.75            1.000
 BRITANNIA  54.15            0.857
       ITC  27.85            0.714
HINDUNILVR  20.07            0.571
     DABUR  18.36            0.429
TATACONSUM   7.57            0.286
  GODREJCP  -4.45            0.143


In [6]:
# Pivot to wide format for Excel — one row per company
# columns = metrics, values = percentile ranks

def generate_peer_excel(peer_df, peer_percentiles, output_path):
    """
    Generates peer_comparison.xlsx with 11 sheets.
    One sheet per peer group with colour-coded percentile cells.
    Green = top quartile (>=75th), Yellow = middle, Red = bottom (<=25th)
    """
    GREEN  = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    YELLOW = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid')
    RED    = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    GOLD   = PatternFill(start_color='FFD700', end_color='FFD700', fill_type='solid')
    HEADER = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')

    wb = Workbook()
    wb.remove(wb.active)

    for group_name in peer_percentiles['peer_group_name'].unique():
        ws = wb.create_sheet(title=group_name[:31])

        # Get companies in this group
        group_companies = peer_df[
            peer_df['peer_group_name'] == group_name
        ]['company_id'].tolist()

        benchmark = peer_df[
            (peer_df['peer_group_name'] == group_name) &
            (peer_df['is_benchmark'] == 1)
        ]['company_id'].tolist()
        benchmark_co = benchmark[0] if benchmark else None

        # Pivot percentiles to wide format
        group_pcts = peer_percentiles[
            peer_percentiles['peer_group_name'] == group_name
        ].pivot(index='company_id', columns='metric', values='percentile_rank')

        group_vals = peer_percentiles[
            peer_percentiles['peer_group_name'] == group_name
        ].pivot(index='company_id', columns='metric', values='value')

        # Write header
        headers = ['company_id'] + [
            f"{m}_value" for m in metric_cols if m in group_vals.columns
        ] + [
            f"{m}_pct_rank" for m in metric_cols if m in group_pcts.columns
        ]
        ws.append(headers)
        for cell in ws[1]:
            cell.fill      = HEADER
            cell.font      = Font(color='FFFFFF', bold=True)
            cell.alignment = Alignment(horizontal='center')

        # Write data rows
        for company in group_companies:
            row = [company]
            for m in metric_cols:
                if m in group_vals.columns:
                    val = group_vals.loc[company, m] if company in group_vals.index else None
                    row.append(round(float(val), 2) if pd.notna(val) else None)
            for m in metric_cols:
                if m in group_pcts.columns:
                    pct = group_pcts.loc[company, m] if company in group_pcts.index else None
                    row.append(round(float(pct), 3) if pd.notna(pct) else None)
            ws.append(row)

        # Colour code percentile rank columns
        pct_start_col = len(metric_cols) + 2
        for row_idx in range(2, ws.max_row + 1):
            company_cell = ws.cell(row=row_idx, column=1)

            # Gold for benchmark company
            if company_cell.value == benchmark_co:
                for col_idx in range(1, ws.max_column + 1):
                    ws.cell(row=row_idx, column=col_idx).fill = GOLD
                continue

            for col_idx in range(pct_start_col, ws.max_column + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                val  = cell.value
                if val is not None:
                    if val >= 0.75:
                        cell.fill = GREEN
                    elif val <= 0.25:
                        cell.fill = RED
                    else:
                        cell.fill = YELLOW

        # Auto width
        for col in ws.columns:
            max_len = max(
                len(str(cell.value)) if cell.value else 0
                for cell in col
            )
            ws.column_dimensions[
                col[0].column_letter
            ].width = min(max_len + 2, 20)

    wb.save(output_path)
    print(f"Saved: {output_path}")
    print(f"Sheets: {wb.sheetnames}")

generate_peer_excel(
    peer_df,
    peer_percentiles,
    'output/peer_comparison.xlsx'
)

Saved: output/peer_comparison.xlsx
Sheets: ['Automobiles', 'Consumer Finance', 'FMCG', 'IT Services', 'Life Insurance', 'Oil & Gas', 'Pharmaceuticals', 'Power & Utilities', 'Private Banks', 'Public Sector Banks', 'Steel']


In [7]:
print("Peer Comparison Summary:")
print(f"  Peer groups: {peer_percentiles['peer_group_name'].nunique()}")
print(f"  Companies:   {peer_percentiles['company_id'].nunique()}")
print(f"  Metrics:     {peer_percentiles['metric'].nunique()}")
print(f"  Total rows:  {len(peer_percentiles)}")
print()
print("Companies per peer group:")
print(peer_df.groupby('peer_group_name')['company_id'].count().to_string())

Peer Comparison Summary:
  Peer groups: 11
  Companies:   56
  Metrics:     10
  Total rows:  560

Companies per peer group:
peer_group_name
Automobiles            7
Consumer Finance       3
FMCG                   7
IT Services            5
Life Insurance         4
Oil & Gas              5
Pharmaceuticals        5
Power & Utilities      7
Private Banks          5
Public Sector Banks    4
Steel                  4
